In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develop a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"  # "20251028_140930" # "20251027_152036"

In [ ]:
"""--------------------------------------------"""
# add in model parameters to encoder, look at cvr2
# cvr2 with strategy model params
# reward prediction error (MF), q-learner, need to get q-value
"""---------------------------------------------"""
# build in interaction terms and see what pops out in the cvr2/dr2 plots
"""---------------------------------------------"""
# add movement over time
"""---------------------------------------------"""
# cvr2 across time
# --> when does encoding emerge across time?
"""---------------------------------------------"""
# aggregate across sessions
"""---------------------------------------------"""
# check for temporal autocorrelations by ensuring no encoding after shuffling trial data.
# incorporate as a sanity check to pass for all sessions

# > check the fits for different regularization constants
# define responsive
"""--------------------------------------------"""

In [ ]:
# TODO
# cvr2/dr2 with bswitch and interaction terms
# plot bswitch bweight over time

## init

In [ ]:
from sg.models import Encoder, StrategyEncoder

encoder = Encoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
        "block_side",
        "response_prev",
        "rewarded_prev",
        "bswitch_itrial",
    ],
    add_interaction=True,
    norm=True,
)
encoder.verify()

encoder_mb = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mb")
encoder_mf = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mf")

encoder_mb.verify()
encoder_mf.verify()

In [ ]:
encoder = Encoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
        "response_prev",
        "rewarded_prev",
    ],
    add_interaction=False,
    norm=True,
)
encoder.verify()

encoder_rp_nan = Encoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
        "response_prev",
        "rewarded_prev",
    ],
    response_prev_only=False,
    add_interaction=False,
    norm=True,
)
encoder_rp_nan.verify()

In [ ]:
from core.viz import plot_kdes

plot_kdes(
    {
        "prev response no nans": encoder.scores["encoder"],
        "prev response nans": encoder_rp_nan.scores["encoder"],
    }
)

In [ ]:
encoder = Encoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
        "response_prev",
        "rewarded_prev",
    ],
    add_interaction=False,
    norm=True,
)
encoder.verify()

encoder_bswitch = Encoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
        "response_prev",
        "rewarded_prev",
        "bswitch_itrial",
    ],
    add_interaction=False,
    norm=True,
)
encoder_bswitch.verify()

In [ ]:
from core.viz import plot_scatter

ax = plot_scatter(
    encoder.scores["encoder"],
    encoder_bswitch.scores["encoder"],
    xlabel="no bswitch",
    ylabel="bswitch",
    add_unity=True,
    add_lr=True,
)
ax.legend(loc="upper left")

In [ ]:
encoder_x = Encoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
        "response_prev",
        "rewarded_prev",
        "bswitch_itrial",
    ],
    add_interaction=True,
    norm=True,
)
encoder_x.verify()

encoder = Encoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
        "response_prev",
        "rewarded_prev",
        "bswitch_itrial",
    ],
    add_interaction=False,
    norm=True,
)
encoder.verify()

In [ ]:
plt.figure()
plt.imshow(encoder_x.dm, interpolation="none", aspect="auto")
plt.colorbar()
plt.show()

In [ ]:
from core.viz import plot_scatter

ax = plot_scatter(
    encoder.scores["encoder"],
    encoder_x.scores["encoder"],
    xlabel="no interaction",
    ylabel="interaction",
    add_unity=True,
    add_lr=True,
)
ax.legend(loc="upper left")

In [ ]:
list(encoder_x.dm_idxs.keys())[36]

In [ ]:
list(encoder_x.dm_idxs.keys())[34]

In [ ]:
encoder_x.fit_encoder()
plt.figure()
plt.imshow(encoder_x.encoder_weights, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.show()

In [ ]:
from itertools import product
import pandas as pd

dm_names = [name for name in encoder.dm_idxs.keys() if "tent" not in name]

pairs = product(dm_names, dm_names)

df = pd.DataFrame([])
for dm_a, dm_b in pairs:
    if (
        dm_a != dm_b
        and not ("bswitch" in dm_a and "bswitch" in dm_b)
        and f"{dm_b}_{dm_a}" not in df.columns
    ):
        df[f"{dm_a}_{dm_b}"] = (
            encoder.dm[:, encoder.dm_idxs[dm_a]] * encoder.dm[:, encoder.dm_idxs[dm_b]]
        )

In [ ]:
import numpy as np

rah = []
names = []
dm_names = [name for name in encoder.dm_idxs.keys() if "tent" not in name]

pairs = product(dm_names, dm_names)
for dm_a, dm_b in pairs:
    print(dm_a != dm_b)
    if (
        dm_a != dm_b
        and not ("bswitch" in dm_a and "bswitch" in dm_b)
        and f"{dm_b}_{dm_a}" not in names
    ):
        print("hai")
        rah.append(
            encoder.dm[:, encoder.dm_idxs[dm_a]] * encoder.dm[:, encoder.dm_idxs[dm_b]]
        )
        names.append(f"{dm_a}_{dm_b}")
rah = np.array(rah)

In [ ]:
rah.shape

In [ ]:
resp = encoder.dm[:, encoder.dm_idxs["response"]]
rewd = encoder.dm[:, encoder.dm_idxs["rewarded"]]

all(resp * rewd == df["response_rewarded"])

In [ ]:
plt.figure()
plt.imshow(encoder.dm, interpolation="none", aspect="auto")
plt.colorbar()
plt.show()

In [ ]:
plt.figure()
plt.imshow(df)
plt.show()

In [ ]:
from sg.models import ShuffledEncoder

se = ShuffledEncoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
        "block_side",
        "strategy",
        "response_prev",
        "rewarded_prev",
    ],
)
se.plot_cvr2()
se.plot_dr2()
se.plot_bound_r2()

## weight correlation

In [ ]:
from core.data import tv_vals
from core.viz import plot_kdes

weight_diff = {}
for regr in encoder.tv_keys:
    if regr != "response_prev":
        regr_ = f"{regr}_{tv_vals[regr][0]}"
        weight_diff[regr_] = (
            encoder_mb.encoder_weights[:, encoder.dm_idxs[regr_]]
            - encoder_mf.encoder_weights[:, encoder.dm_idxs[regr_]]
        )

plot_kdes(weight_diff)

## pca on DMS/DLS robs

In [ ]:
import numpy as np

In [ ]:
from sklearn.decomposition import PCA
from utils.colors import colors_region

pca_dls = PCA().fit(encoder.robs[:, encoder.reg_idxs["DLS"]])
pca_dms = PCA().fit(encoder.robs[:, encoder.reg_idxs["DMS"]])
cum_var_dls = np.array([sum(pca_dls.explained_variance_ratio_[:n]) for n in range(100)])
cum_var_dms = np.array([sum(pca_dms.explained_variance_ratio_[:n]) for n in range(100)])

plt.figure(tight_layout=True)
plt.plot(
    cum_var_dls,
    color=colors_region["DLS"],
    label=f"DLS (n={np.where(cum_var_dls > 0.9)[0][0]})",
)
plt.plot(
    cum_var_dms,
    color=colors_region["DMS"],
    label=f"DMS (n={np.where(cum_var_dms > 0.9)[0][0]})",
)
plt.legend()
plt.axhline(y=0.9, color="#666666", linestyle="--")
plt.xlabel("n. components")
plt.ylabel("p(explained variance)")
plt.title("spike counts")
plt.show()

## pca based on weights

In [ ]:
from sklearn.decomposition import PCA

pca_dls = PCA().fit(encoder.encoder_weights[encoder.reg_idxs["DLS"]])
pca_dms = PCA().fit(encoder.encoder_weights[encoder.reg_idxs["DMS"]])

cum_var_dls = np.array(
    [
        sum(pca_dls.explained_variance_ratio_[:n])
        for n in range(encoder.num_tents + encoder.num_tv)
    ]
)
cum_var_dms = np.array(
    [
        sum(pca_dms.explained_variance_ratio_[:n])
        for n in range(encoder.num_tents + encoder.num_tv)
    ]
)

plt.figure(tight_layout=True)
plt.plot(
    cum_var_dls,
    color=colors_region["DLS"],
    label=f"DLS (n={np.where(cum_var_dls > 0.9)[0][0]})",
)
plt.plot(
    cum_var_dms,
    color=colors_region["DMS"],
    label=f"DMS (n={np.where(cum_var_dms > 0.9)[0][0]})",
)
plt.axhline(y=0.9, color="#666666", linestyle="--")
plt.xlabel("n. components")
plt.ylabel("p(explained variance)")
plt.title("encoding beta weights")
plt.legend()
plt.show()

In [ ]:
pca = PCA().fit(encoder.encoder_weights)
cum_var = np.array(
    [
        sum(pca.explained_variance_ratio_[:n])
        for n in range(encoder.num_tents + encoder.num_tv)
    ]
)

plt.figure(tight_layout=True)
plt.plot(cum_var)
plt.axhline(y=0.9, color="#666666", linestyle="--")
plt.xlabel("n. components")
plt.ylabel("p(explained variance)")
plt.show()

In [ ]:
n = np.where(cum_var >= 0.9)[0][0]
pca = PCA(n_components=n).fit(encoder.encoder_weights)
weights_lowd = pca.transform(encoder.encoder_weights)

In [ ]:
weights_lowd[:, :3]

In [ ]:
plt.figure()
plt.scatter(weights_lowd[:, 0], weights_lowd[:, 1], alpha=0.5, s=0.5)
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection="3d")

ax.scatter(xs=weights_lowd[:, 0], ys=weights_lowd[:, 1], zs=weights_lowd[:, 2], s=0.3)
plt.show()